In [6]:
import os
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.checkpoint.memory import MemorySaver
from uuid import uuid4
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, AIMessage


memory = MemorySaver()

In [7]:
"""
In previous examples we've annotated the `messages` state key
with the default `operator.add` or `+` reducer, which always
appends new messages to the end of the existing messages array.

Now, to support replacing existing messages, we annotate the
`messages` key with a customer reducer function, which replaces
messages with the same `id`, and appends them otherwise.
"""
def reduce_messages(left: list[AnyMessage], right: list[AnyMessage]) -> list[AnyMessage]:
    # assign ids to messages that don't have them
    for message in right:
        if not message.id:
            message.id = str(uuid4())
    # merge the new messages with the existing messages
    merged = left.copy()
    for message in right:
        for i, existing in enumerate(merged):
            # replace any existing messages with the same id
            if existing.id == message.id:
                merged[i] = message
                break
        else:
            # append any new messages to the end
            merged.append(message)
    return merged

class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], reduce_messages]

In [8]:
tool = TavilySearchResults(max_results=2)

In [9]:
class Agent:
    def __init__(self, model, tools, system="", checkpointer=None):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(
            checkpointer=checkpointer,
            interrupt_before=["action"]
        )
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        print(state)
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [10]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(
    model="google/gemini-2.5-flash-lite",
    base_url="https://openrouter.ai/api/v1", 
    openai_api_key=os.getenv("OPENROUTER_API_KEY")) 

abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [11]:
messages = [HumanMessage(content="Whats the weather in SF?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='a42cc972-c2fc-4f64-96d4-cf50de6ba22f'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 122, 'total_tokens': 136, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 1.78e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.78e-05, 'upstream_inference_prompt_cost': 1.22e-05, 'upstream_inference_completions_cost': 5.6e-06}}, 'model_provider': 'openai', 'model_name': 'google/gemini-2.5-flash-lite', 'system_fingerprint': None, 'id': 'gen-1769629658-F4GoNtPVMhrMphspkFn6', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c0625-ab47-7a4

In [12]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='a42cc972-c2fc-4f64-96d4-cf50de6ba22f'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 122, 'total_tokens': 136, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 1.78e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.78e-05, 'upstream_inference_prompt_cost': 1.22e-05, 'upstream_inference_completions_cost': 5.6e-06}}, 'model_provider': 'openai', 'model_name': 'google/gemini-2.5-flash-lite', 'system_fingerprint': None, 'id': 'gen-1769629658-F4GoNtPVMhrMphspkFn6', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_r

In [13]:
abot.graph.get_state(thread).next

('action',)

### continue after interrupt

In [14]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)

Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'weather in San Francisco'}, 'id': 'tool_tavily_search_results_json_79WrwE2nhsZWk7ZgGkM8', 'type': 'tool_call'}
Back to the model!
{'messages': [ToolMessage(content='[{\'title\': \'Weather in San Francisco in January 2026 (California)\', \'url\': \'https://world-weather.info/forecast/usa/san_francisco/january-2026/\', \'content\': "Weather\\n Archive\\n Weather Widget\\n\\n World\\n United States\\n California\\n Weather in San Francisco\\n\\n# Weather in San Francisco in January 2026\\n\\nSan Francisco Weather Forecast for January 2026 is based on long term prognosis and previous years\' statistical data.\\n\\nJanFebMarAprMayJunJulAugSepOctNovDec\\n\\n  \\n\\n## January\\n\\nStart Week On\\n\\n Sun\\n Mon\\n Tue\\n Wed\\n Thu\\n Fri\\n Sat\\n\\n +59°\\n\\n  5.8 mph N 29.8 inHg95 %07:25 AM05:01 PM\\n +61°\\n\\n  8.5 mph SE 29.9 inHg92 %07:25 AM05:02 PM\\n +61°\\n\\n  13.9 mph S 29.8 inHg89 %07:25 AM05:03 PM\\n +55°\\n\\n

In [15]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in SF?', additional_kwargs={}, response_metadata={}, id='a42cc972-c2fc-4f64-96d4-cf50de6ba22f'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 122, 'total_tokens': 136, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 1.78e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.78e-05, 'upstream_inference_prompt_cost': 1.22e-05, 'upstream_inference_completions_cost': 5.6e-06}}, 'model_provider': 'openai', 'model_name': 'google/gemini-2.5-flash-lite', 'system_fingerprint': None, 'id': 'gen-1769629658-F4GoNtPVMhrMphspkFn6', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_r

In [16]:
abot.graph.get_state(thread).next

()

In [17]:
messages = [HumanMessage("Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)
while abot.graph.get_state(thread).next:
    print("\n", abot.graph.get_state(thread),"\n")
    _input = input("proceed?")
    if _input != "y":
        print("aborting")
        break
    for event in abot.graph.stream(None, thread):
        for v in event.values():
            print(v)

{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='75cabdd9-1fdd-43ad-a59e-1a1360c45989'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 122, 'total_tokens': 135, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 1.74e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.74e-05, 'upstream_inference_prompt_cost': 1.22e-05, 'upstream_inference_completions_cost': 5.2e-06}}, 'model_provider': 'openai', 'model_name': 'google/gemini-2.5-flash-lite', 'system_fingerprint': None, 'id': 'gen-1769629748-lVrKs2qrqiNhemLWfZE9', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c0627-0d4a-7f0

## Modify State
Run until the interrupt and then modify the state.

In [24]:
messages = [HumanMessage("Whats the weather in LA?")]
thread = {"configurable": {"thread_id": "3"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='9886900b-ec0e-4c00-beff-f819e7ba8c1c'), AIMessage(content='You need to ask me for a specific location, including the state and country, to get accurate weather information. What specific location in Los Angeles are you interested in?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 122, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 2.54e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 2.54e-05, 'upstream_inference_prompt_cost': 1.22e-05, 'upstream_inference_completions_cost': 1.32e-05}}, 'model_provider': 'openai', 'model_name': 'google/g

In [25]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='9886900b-ec0e-4c00-beff-f819e7ba8c1c'), AIMessage(content='You need to ask me for a specific location, including the state and country, to get accurate weather information. What specific location in Los Angeles are you interested in?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 122, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 2.54e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 2.54e-05, 'upstream_inference_prompt_cost': 1.22e-05, 'upstream_inference_completions_cost': 1.32e-05}}, 'model_provider': 'openai', 'm

In [26]:
current_values = abot.graph.get_state(thread)

In [27]:
current_values.values['messages'][-1]

AIMessage(content='You need to ask me for a specific location, including the state and country, to get accurate weather information. What specific location in Los Angeles are you interested in?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 161, 'total_tokens': 194, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 2.93e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 2.93e-05, 'upstream_inference_prompt_cost': 1.61e-05, 'upstream_inference_completions_cost': 1.32e-05}}, 'model_provider': 'openai', 'model_name': 'google/gemini-2.5-flash-lite', 'system_fingerprint': None, 'id': 'gen-1769629911-pvP5j73tyxKrtKAaCx2y', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run-

In [29]:
current_values.values['messages'][-1].tool_calls

[]

In [23]:
_id = current_values.values['messages'][-1].tool_calls[0]['id']
current_values.values['messages'][-1].tool_calls = [
    {'name': 'tavily_search_results_json',
  'args': {'query': 'current weather in Louisiana'},
  'id': _id}
]

IndexError: list index out of range

In [30]:
abot.graph.update_state(thread, current_values.values)

{'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='9886900b-ec0e-4c00-beff-f819e7ba8c1c'), AIMessage(content='You need to ask me for a specific location, including the state and country, to get accurate weather information. What specific location in Los Angeles are you interested in?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 122, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 2.54e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 2.54e-05, 'upstream_inference_prompt_cost': 1.22e-05, 'upstream_inference_completions_cost': 1.32e-05}}, 'model_provider': 'openai', 'model_name': 'google/g

{'configurable': {'thread_id': '3',
  'checkpoint_ns': '',
  'checkpoint_id': '1f0fc82e-0bf3-6a42-8005-bcc8a37b2639'}}

In [31]:
abot.graph.get_state(thread)

StateSnapshot(values={'messages': [HumanMessage(content='Whats the weather in LA?', additional_kwargs={}, response_metadata={}, id='9886900b-ec0e-4c00-beff-f819e7ba8c1c'), AIMessage(content='You need to ask me for a specific location, including the state and country, to get accurate weather information. What specific location in Los Angeles are you interested in?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 122, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 2.54e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 2.54e-05, 'upstream_inference_prompt_cost': 1.22e-05, 'upstream_inference_completions_cost': 1.32e-05}}, 'model_provider': 'openai', 'm

In [32]:
for event in abot.graph.stream(None, thread):
    for v in event.values():
        print(v)